# 4 Searching architectures, and what a search may conclude

`1_representation_and_head/04_search` ran one Optuna search and read it, and
`03_what_does_the_algorithm_know` compared what the samplers know. This notebook runs a
search with one of them, TPE, on the composition family of `01_gates_highways_moe` -- gate,
highway, dense and sparse mixture of experts -- on `solar_10_minutes`: 20 solar sites pooled
into one training set, a daily cycle that is deterministically zero every night. A pool of
different sites is at least plausibly something a router could divide up. The questions are
whether a search finds out, and what it may conclude once it has.

At the end you can:

- read a learning-rate schedule over epochs, say what a smaller learning rate does to each
  update of the weights, and tell a scheduler (it changes the learning rate within one
  training) from a pruner (it stops whole trials);
- read a Pareto front over score, number of weights and speed, choose a model from it under
  a concrete bound on parameters or on speed, and say whether the chosen model's quality
  differs from the best by more than the seeds can show;
- count a search space and say why it grows exponentially with the number of axes while a
  budget grows by one trial at a time; say how successive halving lets trials compete for
  training budget, when stopping early saves budget, and why a configuration that learns late
  can drop out wrongly.

Four things come on top of lesson 1's search: a baseline named before the search (4.1), a
learning-rate schedule as a lever of its own (4.2), a capacity-matched control inside the
search space, a choice under a bound and a confirmation of the winners over new seeds, with
the test windows read once at the end (4.4), and a pruner that stops weak trials early (4.5).

In [ ]:
import functools
import itertools
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import datasets
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
from loguru import logger
from notebooktester import param
from optuna.trial import TrialState
from solar_10_minutes import load_solar
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from vectormesh.components import Gate, Highway, MoE, NeuralNet, SparseMoE
from vectormesh.components.metrics import MAE
from vectormesh.training import Trainer, TrainerSettings
from windowing import window_index

logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()
# a search shows one progress bar; Optuna's line per trial stays off
optuna.logging.set_verbosity(optuna.logging.WARNING)

# one window: W = 288 steps of 10 minutes (two days) of history in, H = 144 steps (the
# next day) out; the window 2_kernels_state_and_gates/03_decomposition_and_nbeats uses
W, H = 288, 144
EPOCHS = param(15, test=1)
# 36 trials: six mechanisms, and two more axes for the routed ones
N_TRIALS = param(36, test=2)
SEEDS = param(5, test=2)
# where a Trainer may write a checkpoint; nothing here saves one, so it is never made
LOGDIR = Path("modellogs").resolve()

## 4.1 The baseline, first

Same discipline every notebook in this course insists on: name the number a search has
to beat before running one. Every model here maps one window to a forecast: the last
`W` = 288 steps of 10 minutes (two days) of a site's power in, the next `H` = 144 steps (one
day) out, both scaled with the mean and spread of that site's training part. The baseline is
the simplest model of the family of 4.3, `plain`: `Linear(288 -> 64) -> GELU -> Linear(64 ->
144)`, hidden width 64, Adam at a constant `lr` of 1e-3, 15 epochs, five seeds, scored by the
val MAE after the last epoch. Its test score is computed too, but read only once, at the end
of 4.4.

**A difference beside its spread.** Every difference below is read in val MAE, beside the
standard error of that difference: `sqrt(std_a**2/n_a + std_b**2/n_b)`, with the std over
seeds (`ddof=1`). When two arms share their seeds run by run (the same initial weights and
the same batches), the difference is taken per seed, and its standard error is that of the
mean of those per-seed differences (paired by seed). `se_of_difference` and `paired_se`
below compute the two.

In [ ]:
@functools.cache
def load_solar_windows(
    w: int = W, h: int = H, n_series: int = 20, stride: int = 6
) -> dict:
    """The split_scale_window of 2_kernels_state_and_gates/03_decomposition_and_nbeats on
    load_solar's output: per site the first 60% of the series is train, the next 20% val,
    the last 20% test, scaled with the train part's mean and std. Kept per window length,
    so a search that draws `w` (E2) cuts every length once."""
    series = load_solar(w, h, n_series=n_series)

    parts = {"train": [[], []], "val": [[], []], "test": [[], []]}
    for s in series:
        n = len(s)
        cuts = {
            "train": (0, int(0.6 * n)),
            "val": (int(0.6 * n), int(0.8 * n)),
            "test": (int(0.8 * n), n),
        }
        mu = s[: cuts["train"][1]].mean()
        sigma = s[: cuts["train"][1]].std().clamp(min=1e-6)
        for split, (a, b) in cuts.items():
            v = (s[a:b] - mu) / sigma
            win = v[window_index(len(v), w + h, stride)]
            parts[split][0].append(win[:, :w])
            parts[split][1].append(win[:, w:])
    return {k: (torch.cat(v[0]), torch.cat(v[1])) for k, v in parts.items()}


power = load_solar_windows()
xtr, ytr = power["train"]
xva, yva = power["val"]
xte, yte = power["test"]
print(f"train {tuple(xtr.shape)}   val {tuple(xva.shape)}   test {tuple(xte.shape)}")


def mae(y: torch.Tensor, yhat: torch.Tensor) -> float:
    return float((y - yhat).abs().mean())


def make_expert(hidden: int, expert_hidden: int) -> nn.Module:
    # a bottleneck 2-layer expert -- Linear -> GELU -> Linear -- with its own width
    # independent of the block's hidden dim. A single Linear (01_gates_highways_moe's
    # expert shape) can only ever affine-remap the shared feature space; a real
    # specialist needs at least one nonlinearity to represent a genuinely different
    # regime, which is the whole premise a router is betting on.
    return NeuralNet(hidden_size=[hidden, expert_hidden], out_size=hidden)


def make_model(
    mechanism: str,
    hidden: int,
    n_experts: int = 4,
    expert_hidden: int | None = None,
    w: int = W,
    h: int = H,
) -> nn.Module:
    # the family 01_gates_highways_moe built: Linear -> GELU -> [mechanism] -> Linear.
    # "plain" has nothing in the middle; "deeper" is the capacity-matched control of
    # "gate", and both are one more choice of the search space (4.3).
    expert_hidden = expert_hidden or hidden
    if mechanism == "plain":
        mid = nn.Identity()
    elif mechanism == "deeper":
        # the capacity-matched control of `gate`: a Linear -> GELU block with exactly
        # the parameters of Gate's weight matrix and bias, and no gating
        mid = nn.Sequential(nn.Linear(hidden, hidden), nn.GELU())
    elif mechanism == "gate":
        mid = Gate(hidden_size=hidden)
    elif mechanism == "highway":
        mid = Highway(transform=NeuralNet(hidden, hidden), hidden_size=hidden)
    elif mechanism == "moe":
        mid = MoE(
            experts=[make_expert(hidden, expert_hidden) for _ in range(n_experts)],
            hidden_size=hidden,
            out_size=hidden,
        )
    elif mechanism == "sparse_moe":
        mid = SparseMoE(
            experts=[make_expert(hidden, expert_hidden) for _ in range(n_experts)],
            hidden_size=hidden,
            out_size=hidden,
            k=1,
        )
    else:
        raise ValueError(mechanism)
    return nn.Sequential(nn.Linear(w, hidden), nn.GELU(), mid, nn.Linear(hidden, h))


def param_counts(model: nn.Module, mechanism: str, k: int = 1) -> tuple[int, int]:
    """(total, active) parameter counts for one forward pass. Every mechanism here
    except sparse_moe runs every parameter on every input, so total == active --
    that equality is itself the point dense mechanisms cannot escape. sparse_moe only
    ever exercises k of its N experts per example: active counts the router, the
    block's other layers, and k experts' worth of weights, not all N."""
    total = sum(p.numel() for p in model.parameters())
    if mechanism != "sparse_moe":
        return total, total
    mid = model[2]
    per_expert = sum(p.numel() for p in mid.experts[0].parameters())
    n_experts = len(mid.experts)
    non_expert = total - n_experts * per_expert
    active = non_expert + k * per_expert
    return total, active


validloader = DataLoader(TensorDataset(xva, yva), batch_size=len(xva))


def run(
    model: nn.Module, epochs: int = EPOCHS, lr: float = 1e-3, batch_size: int = 64
) -> tuple[float, float]:
    """Train for `epochs` epochs without early stopping; return the val MAE after the last
    epoch and the test MAE of the same weights."""
    # one Trainer per run: progress=False drops the tqdm bars, and nothing is written
    # to LOGDIR, so constructing one per seed costs nothing
    trainloader = DataLoader(
        TensorDataset(xtr, ytr), batch_size=batch_size, shuffle=True
    )
    settings = TrainerSettings(
        epochs=epochs,
        metrics=[MAE()],
        logdir=LOGDIR,
        train_steps=len(trainloader),
        valid_steps=len(validloader),
        optimizer_kwargs={"lr": lr},
        earlystop_kwargs=None,
    )
    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=nn.MSELoss(),
        optimizer=torch.optim.Adam,
        traindataloader=trainloader,
        validdataloader=validloader,
        progress=False,
    )
    val_mae = trainer.loop().metric_dict["MAE"]
    model.eval()
    with torch.no_grad():
        return val_mae, mae(yte, model(xte))


@dataclass
class Spread:
    """The scores of one configuration over several seeds."""

    values: list[float]

    @property
    def n(self) -> int:
        return len(self.values)

    @property
    def mean(self) -> float:
        return float(np.mean(self.values))

    @property
    def std(self) -> float:
        """The spread between seeds (ddof=1), as in lesson 2."""
        return float(np.std(self.values, ddof=1)) if self.n > 1 else 0.0

    def __str__(self) -> str:
        return f"{self.mean:.4f} +/- {self.std:.4f}"


def se_of_difference(a: Spread, b: Spread) -> float:
    """The standard error of mean_a - mean_b for two arms with runs of their own:
    sqrt(std_a**2 / n_a + std_b**2 / n_b)."""
    return float(np.sqrt(a.std**2 / a.n + b.std**2 / b.n))


def paired_se(a: Spread, b: Spread) -> float:
    """The standard error of mean_a - mean_b for two arms that share their seeds run by
    run (the same initial weights, the same batches): the standard error of the mean of
    the per-seed differences."""
    d = np.asarray(a.values) - np.asarray(b.values)
    return float(d.std(ddof=1) / np.sqrt(len(d)))


val_scores, test_scores = [], []
start = time.time()
for seed in range(SEEDS):
    torch.manual_seed(seed)
    val_mae, test_mae = run(make_model("plain", hidden=64))
    val_scores.append(val_mae)
    test_scores.append(test_mae)

BASELINE = Spread(val_scores)
# the test score waits for the one report at the end of 4.4
BASELINE_TEST = Spread(test_scores)
print(f"seeds: {[f'{s:.4f}' for s in val_scores]}")
print(
    f"\nBASELINE (plain, hidden=64, lr 1e-3)  val {BASELINE}   "
    f"({time.time() - start:.0f}s for {SEEDS} runs)"
)

## 4.2 The learning rate: coarse first, then fine

The baseline above holds `lr` at 1e-3 for every epoch, and the search below treats `lr` as
one more number to choose. Lesson 1.7 called the learning rate the step size: plain SGD
moves every weight by its gradient times `lr`. Adam divides a running *average* of each
weight's gradient by the running *size* of that gradient (the root of its mean square), so
the slope's steepness cancels out. Gradients point a little differently from batch to batch
and partly cancel in the average, while the size counts every batch whatever its sign, so the
ratio is usually well below 1: on average a weight moves by `lr` times a factor below 1 in one
update (the cell below measures it), though a single weight can now and then move more than
`lr`. **A smaller learning rate makes every update shorter, in the same direction**:
within one training, halve `lr` and each weight moves half as far per batch.

The right step size changes during a run. A big step gets near a good solution quickly and
then keeps bouncing around it; a small step settles precisely but needs many epochs to
arrive. A **learning-rate scheduler** changes `lr` while one training runs: the chainsaw
first, for the shape, the chisel after, for the detail. **Cosine decay**
(`torch.optim.lr_scheduler.CosineAnnealingLR`) is the smoothest version: `lr` starts at its
maximum and follows half a cosine wave down towards zero over `T_max` epochs.

In the Trainer a scheduler is one extra argument, `scheduler=CosineAnnealingLR`, with
`scheduler_kwargs={"T_max": EPOCHS}` for the length of the half cosine. The Trainer steps it
once per epoch, after that epoch's validation, so the first epoch trains at the start value.
The cell below does not compute the schedule from a formula: it reads `lr` from the
optimizer before every epoch, so the plot shows what each arm trained with. It also measures
how far the weights move in one update: the mean absolute change of a weight in one
`optimizer.step()`, averaged over the epoch.

Four arms, the same model as the baseline (`plain`, hidden=64), the same 15 epochs, the
same seeds, no early stopping, scored on the validation windows:

| arm | `lr` during training | what it isolates |
|---|---|---|
| `constant 1e-3` | 1e-3 at every epoch | the baseline above |
| `cosine 1e-3` | 1e-3, falling towards 0 | the decay alone: same start as the baseline |
| `constant 3e-3` | 3e-3 at every epoch | a bigger step, no decay (3e-3 is the top of the `lr` range the search below uses) |
| `cosine 3e-3` | 3e-3, falling towards 0 | the bigger start with the decay |

Each cosine arm has a constant arm with the same start, so every pair differs in the decay
only. A scheduler changes only the size of the steps inside one training: every arm trains
all its 15 epochs. (4.5 meets a pruner, which stops whole trials of a search; that is a
different lever.)

Before you run it, write down: where is the `lr` of `cosine 3e-3` at epoch 8, and at epoch
15? How should the update size of a cosine arm change over the epochs, and of a constant arm?
For which start do you expect the decay to help most, and does the bigger start (the
chainsaw) end lower than the small one once both decay?

In [ ]:
def lr_run(lr: float, cosine: bool, seed: int) -> dict[str, list[float]]:
    """Train `plain` (hidden=64) for EPOCHS epochs starting at `lr`, with or without
    cosine decay over those epochs. Returns per epoch the val MAE after it (`val`), the lr
    the optimizer trained it with (`lr`, read from the optimizer), and the mean absolute
    change of a weight in one update during it (`update`)."""
    torch.manual_seed(seed)
    model = make_model("plain", hidden=64)
    trainloader = DataLoader(TensorDataset(xtr, ytr), batch_size=64, shuffle=True)
    weights = list(model.parameters())
    n_weights = sum(p.numel() for p in weights)
    val_curve: list[float] = []
    update_curve: list[float] = []
    before: list[torch.Tensor] = []
    step_sizes: list[float] = []

    # around every optimizer.step(): copy the weights before, measure how far they moved
    def keep_weights(optimizer, args, kwargs):
        before[:] = [p.detach().clone() for p in weights]

    def measure_update(optimizer, args, kwargs):
        moved = sum(
            float((p.detach() - b).abs().sum()) for p, b in zip(weights, before)
        )
        step_sizes.append(moved / n_weights)

    # the Trainer steps the scheduler after an epoch's validation and then calls the
    # reporters, so here the optimizer already holds the lr of the next epoch
    def track(epoch, train_loss, test_loss, metric_dict):
        val_curve.append(metric_dict["MAE"])
        update_curve.append(float(np.mean(step_sizes)))
        step_sizes.clear()
        lr_curve.append(trainer.optimizer.param_groups[0]["lr"])

    settings = TrainerSettings(
        epochs=EPOCHS,
        metrics=[MAE()],
        logdir=LOGDIR,
        train_steps=len(trainloader),
        valid_steps=len(validloader),
        optimizer_kwargs={"lr": lr},
        scheduler_kwargs={"T_max": EPOCHS},
        earlystop_kwargs=None,
    )
    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=nn.MSELoss(),
        optimizer=torch.optim.Adam,
        scheduler=torch.optim.lr_scheduler.CosineAnnealingLR if cosine else None,
        traindataloader=trainloader,
        validdataloader=validloader,
        reporters=[track],
        progress=False,
    )
    trainer.optimizer.register_step_pre_hook(keep_weights)
    trainer.optimizer.register_step_post_hook(measure_update)
    lr_curve = [trainer.optimizer.param_groups[0]["lr"]]
    trainer.loop()
    # the last entry is the lr after the last epoch, which no epoch trains with
    return {"val": val_curve, "lr": lr_curve[:-1], "update": update_curve}


LR_ARMS = {
    "constant 1e-3": (1e-3, False),
    "cosine 1e-3": (1e-3, True),
    "constant 3e-3": (3e-3, False),
    "cosine 3e-3": (3e-3, True),
}
lr_results: dict[str, dict] = {}
for name, (lr0, cosine) in LR_ARMS.items():
    arm_runs = [lr_run(lr0, cosine, seed) for seed in range(SEEDS)]
    curves = np.array([r["val"] for r in arm_runs])
    lr_results[name] = {
        "curves": curves,
        "lr": np.array(arm_runs[0]["lr"]),
        "update": np.array([r["update"] for r in arm_runs]).mean(0),
    }
    lr_results[name]["last"] = Spread(list(curves[:, -1]))
    mean_curve = curves.mean(0)
    print(
        f"{name:14s} val MAE at epoch {EPOCHS}: {lr_results[name]['last']}   "
        f"best epoch of the mean curve: {mean_curve.argmin() + 1} ({mean_curve.min():.4f})"
    )

# an arm and its partner share the seed of every run: the same initial weights, the
# same batches, so the comparison is paired by seed
print(
    "\npairs at the last epoch, the difference and its standard error (paired by seed):"
)
for a, b in [
    ("cosine 1e-3", "constant 1e-3"),
    ("cosine 3e-3", "constant 3e-3"),
    ("cosine 1e-3", "cosine 3e-3"),
]:
    ra, rb = lr_results[a]["last"], lr_results[b]["last"]
    print(f"  {a} - {b}: {ra.mean - rb.mean:+.4f}, se {paired_se(ra, rb):.4f}")

print("\nthe lr each epoch of cosine 3e-3 trained with, read from its optimizer:")
for epoch, value in enumerate(lr_results["cosine 3e-3"]["lr"], start=1):
    print(f"  epoch {epoch:2d}: {value:.2e}")

print(f"\n{'arm':14s} {'update, epoch 1':>16s} {f'epoch {EPOCHS}':>10s}   update / lr")
for name, r in lr_results.items():
    ratio = r["update"] / r["lr"]
    print(
        f"{name:14s} {r['update'][0]:>16.2e} {r['update'][-1]:>10.2e}   "
        f"{ratio[0]:.2f} -> {ratio[-1]:.2f}"
    )

In [ ]:
fig, (ax_lr, ax_step, ax_mae) = plt.subplots(1, 3, figsize=(14, 3.6))
epochs_axis = np.arange(1, EPOCHS + 1)
for name, r in lr_results.items():
    ax_lr.plot(epochs_axis, r["lr"], marker="o", ms=3, label=name)
    ax_step.plot(epochs_axis, r["update"], marker="o", ms=3, label=name)
    mean, std = r["curves"].mean(0), r["curves"].std(0)
    ax_mae.plot(epochs_axis, mean, label=name)
    ax_mae.fill_between(epochs_axis, mean - std, mean + std, alpha=0.2)
for ax in (ax_lr, ax_step):
    ax.set_yscale("log")
    ax.set_xlabel("epoch")
ax_lr.set_ylabel("lr (log)")
ax_lr.set_title("the lr each epoch trained with")
ax_step.set_ylabel("mean |change| of a weight per update (log)")
ax_step.set_title("the size of one update follows the lr")
ax_mae.set_xlabel("epoch")
ax_mae.set_ylabel("val MAE (mean, band = 1 std over seeds)")
ax_mae.set_title("what the lr does to the curve")
ax_mae.legend(fontsize=8)
plt.tight_layout()

**The schedule itself.** Read from its optimizer, `cosine 3e-3` trains epoch 1 at 3.00e-3,
epoch 8 at 1.66e-3 (a little over half) and epoch 15 at 3.3e-5. The last value is not zero:
the half cosine reaches zero after `T_max` = 15 steps of the scheduler, and the 15th epoch
trains before that last step. The constant arms hold their `lr` at every epoch.

**The update follows the learning rate.** In `constant 1e-3` a weight moves on average 1.66e-4
per update, from the first epoch to the last: about a sixth of the `lr`. In `cosine 1e-3` it
starts at the same 1.66e-4 and falls with the `lr` to 2.0e-6 at epoch 15. The ratio of update
to `lr` stays between 0.13 and 0.18 in every arm while the `lr` of a cosine arm falls about a
hundredfold, and in the middle panel each update curve follows the shape of its `lr` curve.
The factor is not constant: it grows a little as the `lr` falls (0.13 to 0.18 in `cosine
3e-3`), and in `constant 3e-3` the update rises from 3.96e-4 to 4.58e-4 at a flat `lr`. But
next to a hundredfold fall of the `lr` those changes are small. So within one training a smaller
learning rate gives a proportionally shorter update: at epoch 15 of a cosine arm each weight
moves less than a fiftieth as far per batch as at epoch 1. Between trainings the factor
shifts a little: tripling the start from 1e-3 to 3e-3 gives 2.4 times the update at epoch 1
(0.13 of the `lr` instead of 0.17), because the larger steps change the gradients Adam
divides by.

Each cosine arm shares its seeds with its constant partner (the same initial weights, the
same batches), so the pairs are compared per seed, each difference beside its standard error:

- **Decay alone, at the baseline's start (`constant 1e-3` against `cosine 1e-3`):** 0.2281 ±
  0.0023 against 0.2165 ± 0.0004, a gain of 0.0117 from changing nothing but the schedule,
  with a standard error of 0.0010. The constant arm is not merely caught on a bad last epoch: the best
  epoch of its mean curve (0.2254, epoch 11) is still far above the cosine arm.
- **Decay with the bigger start (`constant 3e-3` against `cosine 3e-3`):** 0.2421 ± 0.0021
  against 0.2204 ± 0.0008, a gain of 0.0218, standard error 0.0011. The constant 3e-3 curve
  never settles: it wobbles around 0.24 from the first epoch to the last, while the decaying
  curve falls steadily as its updates shrink. A large step finds the neighbourhood and then
  keeps stepping over the bottom of it; the small steps at the end are what resolve the
  detail.
- **The bigger start did not pay at this budget.** `cosine 1e-3` (0.2165) ends below
  `cosine 3e-3` (0.2204), a gap of 0.0039, standard error 0.0003. Fifteen epochs are enough for
  the small start to arrive, so the chainsaw has no time to save. Whether a bigger start wins
  with a larger epoch budget is not measured here (E5).
- **Both decaying curves are still falling at epoch 15** (their best epoch is the last), while
  the constant curves stopped improving (best epochs 11 and 6). At epoch 15 the `lr` of a
  cosine arm is about 1% of its start, so "still falling" does not mean more epochs would help:
  the schedule is over. `cosine` ends at a fixed epoch because `T_max` is set to it, so it
  cannot be combined with early stopping on a free-running budget the way the runs in
  `01_gates_highways_moe` are.

Two consequences for what follows. The baseline of 4.1 (0.2281) is the number to beat at a
constant `lr`, and a schedule alone moves `plain` by 0.0117, a gain of the same order as the
differences between mechanisms that 4.4 reports. The search below keeps `lr` constant in
every trial, so each trial is compared with the others at the same `lr` policy, and none of
them has the decay.

## 4.3 The search space

`mechanism` is the axis this notebook exists to test: `plain` (no block in the middle),
`deeper` (a `Linear -> GELU` block with exactly the parameters of `gate` and no gating: the
capacity-matched control, folded into the search as one more choice), `gate`, `highway`,
`moe` and `sparse_moe`. `hidden` and `lr` are in the space because width and learning rate
can confound a comparison between architectures if they are not searched alongside it.
`n_experts` and `expert_hidden` only matter for `moe` and `sparse_moe`. Optuna can draw an
axis conditionally, only when another axis has a given value; a `Space` of `hpo` keeps it
simple and draws every axis in every trial, so the other mechanisms ignore these two (their
trials differ in a number that does nothing). An expert is a `Linear -> GELU -> Linear` network with its own width
`expert_hidden`, so the search can trade many small experts against few large ones.

**The sampler.** `TPESampler`, the TPE of `03_what_does_the_algorithm_know`: its first
`n_startup_trials=10` trials are drawn at random; after that it splits the trials so far into
a good pile and a bad pile and draws the next configuration where the good ones are dense
compared with the bad ones. The good pile is small: Optuna puts the best tenth of the trials
so far in it, rounded up, so 1 trial when it proposes trial 10, 2 at trial 11 to 20, and at
most 4 in this search. At trial 10 the good pile is the single best of the ten random
trials. It learns from earlier trials, so it spends more of the later
trials on the mechanisms that scored well early, a point that matters when the results are
read (4.4). `seed=0` makes it propose the same first ten configurations on every run of this notebook,
and the same later ones as long as the scores come out the same (on another machine they can
differ in the last digits);
`search` deletes a study of the same name before it starts, so every run begins on an empty
study and TPE learns only from this run's trials.

**What a trial reports.** After every epoch a trial reports its val MAE to Optuna with
`trial.report`, so the study holds the whole curve of every trial, not only its end. The
study maximises its value, so the value is minus the MAE: higher is better, and the tables
below undo the minus. The curves are what the pruner of 4.5 decides on.

**Why this might not help, and what would make it help.** Routing only pays for its
parameters if the data contains distinct situations that benefit from different transforms:
a cloudy-Tuesday site against a clear-sky one. Pooling 20 sites is at least *plausibly* that
kind of heterogeneity, unlike a single homogeneous series. Whether it pays is what the search
checks, not what this paragraph assumes.

**The trade-off the search reports.** Every non-routed mechanism trains and uses the same
number of parameters. `sparse_moe` can *train* far more than it *uses* on one input (4.4
reports both numbers), the bet `01_gates_highways_moe` measures in 1.4 and 1.5.

**The size of the space against the budget.** Count the configurations. The four discrete
axes alone give 6 × 3 × 6 × 3 = 324 combinations, and `lr` is continuous on top. Not all of
them are different models: for the four mechanisms without experts the two expert axes do
nothing, which leaves 4 × 3 + 2 × 3 × 6 × 3 = 120 distinct architectures, each at any `lr`.
Against that the budget is 36 trials. Every axis you add *multiplies* the space: the window
length of E2, three values, makes it 972 combinations (360 architectures), while each trial
you can afford *adds* one. With k axes of three values each the space has 3^k points: it
grows exponentially with the number of axes, while the budget grows by one trial at a time. A
search can therefore never cover a space of this size; it
samples it, and TPE samples it unevenly on purpose. That is why the budget is spent with
care: on a sampler that learns where to look (4.3), and on stopping trials early so more of
them fit (4.5).

**The budget.** 36 trials of 15 epochs each is 540 epochs of training. A trial's epochs are
fixed rather than early-stopped: a search ranks configurations under an equal short budget,
so the ranking it gives is the ranking *at 15 epochs*, and a configuration that would
overtake later is not seen. That is a legitimate economy as long as it is said; the numbers
reported afterwards are confirmed in 4.4. The study is `lesson3-search` in the course's
Optuna store.

In [ ]:
from hpo.measure import Curves
from hpo.record import SESSION, RunRecord, StudyLog
from hpo.record.store import Store
from hpo.search import Choice, Float, Space

SPACE = Space(
    {
        "mechanism": Choice(
            ["plain", "deeper", "gate", "highway", "moe", "sparse_moe"]
        ),
        "hidden": Choice([32, 64, 128]),
        "n_experts": Choice([2, 4, 6, 8, 10, 16]),  # ignored unless moe/sparse_moe
        "expert_hidden": Choice([16, 32, 64]),  # ignored unless moe/sparse_moe
        "lr": Float(1e-4, 3e-3, log=True),
        "w": W,
        "h": H,
        "epochs": EPOCHS,
    }
)

LATENCY_REPEATS = param(20, test=2)


def latency_ms(model: nn.Module) -> float:
    """The median wall time, in ms, of one forward pass over all validation windows at
    once, after one pass to warm up (4.4 explains why it is measured)."""
    model.eval()
    times = []
    with torch.no_grad():
        model(xva)
        for _ in range(LATENCY_REPEATS):
            start = time.perf_counter()
            model(xva)
            times.append(time.perf_counter() - start)
    return 1e3 * float(np.median(times))


def experts_used(model: nn.Module) -> int:
    """How many experts of a trained sparse_moe get at least one validation window: the
    router's choice (k=1) for every window, counted per expert."""
    model.eval()
    with torch.no_grad():
        hidden = model[1](model[0](xva))
        choice = model[2].router(hidden).argmax(dim=-1)
    return int((torch.bincount(choice, minlength=len(model[2].experts)) > 0).sum())


def objective(trial: optuna.Trial, log: StudyLog) -> float:
    """One trial: draw a config from SPACE, train it for its epochs, report the val MAE
    after every epoch, and return it after the last. The study maximises its value, so
    the value is minus the MAE: higher is better. After every report the pruner may stop
    the trial (4.5); without one, every trial runs all its epochs."""
    config = SPACE.draw(trial)
    data = load_solar_windows(config["w"], config["h"])
    xtr, ytr = data["train"]
    xva, yva = data["val"]

    torch.manual_seed(0)
    model = make_model(
        config["mechanism"],
        config["hidden"],
        config["n_experts"],
        config["expert_hidden"],
        config["w"],
        config["h"],
    )
    total_params, active_params = param_counts(model, config["mechanism"])
    curves = Curves()
    val_mae: list[float] = []

    # one report per epoch, step = the epochs trained so far: a pruner needs the partial
    # curve of every trial to decide which ones to stop
    def to_optuna(epoch, train_loss, test_loss, metric_dict):
        curves.add(train_loss, test_loss, None)
        val_mae.append(metric_dict["MAE"])
        trial.report(-metric_dict["MAE"], epoch + 1)
        if trial.should_prune():
            raise optuna.TrialPruned()

    trainloader = DataLoader(TensorDataset(xtr, ytr), batch_size=64, shuffle=True)
    validloader = DataLoader(TensorDataset(xva, yva), batch_size=len(xva))
    settings = TrainerSettings(
        epochs=config["epochs"],
        metrics=[MAE()],
        logdir=LOGDIR,
        train_steps=len(trainloader),
        valid_steps=len(validloader),
        optimizer_kwargs={"lr": config["lr"]},
        earlystop_kwargs=None,
    )
    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=nn.MSELoss(),
        optimizer=torch.optim.Adam,
        traindataloader=trainloader,
        validdataloader=validloader,
        reporters=[to_optuna],
        progress=False,  # the study's own bar shows progress across trials
    )
    start = time.time()
    trained_ms = None
    n_used = None
    try:
        trainer.loop()
        # the time of a sparse_moe depends on its trained router (4.4), so it is
        # timed here, on the trained model
        if config["mechanism"] == "sparse_moe":
            trained_ms = latency_ms(model)
            n_used = experts_used(model)
    finally:
        # a stopped trial is recorded too: what it drew, and how far it got
        log.record(
            trial,
            RunRecord(
                role="trial",
                dataset="solar_10_minutes",
                config=config,
                fixed=SPACE.fixed(),
                train_seconds=time.time() - start,
                curves=curves,
                extra={
                    "val_mae": round(val_mae[-1], 5),
                    "epochs_trained": len(val_mae),
                    "total_params": total_params,
                    "active_params": active_params,
                    # only a finished sparse_moe has a trained time
                    **(
                        {"trained_ms": trained_ms, "experts_used": n_used}
                        if trained_ms is not None
                        else {}
                    ),
                },
            ),
        )
    return -val_mae[-1]


def search(
    name: str, pruner: optuna.pruners.BasePruner | None = None
) -> tuple[StudyLog, float]:
    """The same search every time, so two searches differ only in `pruner`: TPE with
    10 random starting trials and seed 0, N_TRIALS trials in all. Returns the study and
    the seconds the trials took."""
    # every run starts on an empty study: TPE's seed then proposes the same trials, and
    # the pruner compares only this run's trials
    store = Store.default()
    if name in optuna.get_all_study_names(store.url):
        optuna.delete_study(study_name=name, storage=store.url)
    log = StudyLog.search(
        name=name,
        space=SPACE,
        sampler=optuna.samplers.TPESampler(n_startup_trials=10, seed=0),
        pruner=pruner,
        attrs={"origin": "3_scaling_and_search/04_search", "value": "minus val MAE"},
    )
    log.study.set_metric_names(["minus val MAE"])
    start = time.time()
    log.optimize(lambda trial: objective(trial, log), n_trials=N_TRIALS)
    return log, time.time() - start


def trials_frame(log: StudyLog) -> pd.DataFrame:
    """One row per trial of this run of the notebook: what it drew, its val MAE after
    the last epoch it trained, and how many epochs that was."""
    rows = []
    for t in log.trials(TrialState.COMPLETE, TrialState.PRUNED):
        if t.user_attrs.get("session") != SESSION:
            continue
        config = t.user_attrs["config"]
        rows.append(
            {
                "mae": t.user_attrs["val_mae"],
                **{
                    k: config[k]
                    for k in ("mechanism", "hidden", "n_experts", "expert_hidden", "lr")
                },
                "total_params": t.user_attrs["total_params"],
                "active_params": t.user_attrs["active_params"],
                "trained_ms": t.user_attrs.get("trained_ms"),
                "experts_used": t.user_attrs.get("experts_used"),
                "epochs": t.user_attrs["epochs_trained"],
                "pruned": t.state == TrialState.PRUNED,
                "trial": t.number,
            }
        )
    return pd.DataFrame(rows)


full, full_seconds = search("lesson3-search")
df = trials_frame(full)
print(f"{len(df)} trials of {EPOCHS} epochs in {full_seconds:.0f}s")

## 4.4 Reading the result

In [ ]:
ROUTED = ["moe", "sparse_moe"]


def shown(frame: pd.DataFrame) -> pd.DataFrame:
    """The frame for reading: `n_experts` and `expert_hidden` as a dash where the
    mechanism has no experts and the two numbers do nothing."""
    out = frame.astype({"n_experts": object, "expert_hidden": object})
    out.loc[~out["mechanism"].isin(ROUTED), ["n_experts", "expert_hidden"]] = "-"
    return out


print(
    shown(df.drop(columns=["epochs", "pruned", "trial", "trained_ms", "experts_used"]))
    .sort_values("mae")
    .head(8)
    .to_string(index=False)
)

print(f"\n{'mechanism':<12} {'trials':>7} {'best':>8} {'median':>8} {'worst':>8}")
print("-" * 47)
for name, group in df.groupby("mechanism"):
    print(
        f"{name:<12} {len(group):>7} {group['mae'].min():>8.4f} "
        f"{group['mae'].median():>8.4f} {group['mae'].max():>8.4f}"
    )
start_trials = df[df["trial"] < 10]
first = start_trials.loc[start_trials["mae"].idxmin()]
print(
    f"\nbest of the {len(start_trials)} random starting trials: trial {int(first['trial'])}, "
    f"{first['mechanism']}, {first['mae']:.4f}"
)
print(f"baseline (plain, hidden=64, {SEEDS} seeds), val: {BASELINE}")

# what sparse_moe trains against what it uses on one input
sparse_trials = df[df["mechanism"] == "sparse_moe"].sort_values("mae")
if len(sparse_trials):
    best_sparse = sparse_trials.iloc[0]
    print(
        f"\nbest sparse_moe trial: n_experts={int(best_sparse['n_experts'])}, "
        f"expert_hidden={int(best_sparse['expert_hidden'])}, mae={best_sparse['mae']:.4f}\n"
        f"  params trained: {int(best_sparse['total_params']):>8,}\n"
        f"  params used per input: {int(best_sparse['active_params']):>8,}  "
        f"({best_sparse['active_params'] / best_sparse['total_params']:.0%} of the trained)"
    )

The search table, trial by trial: the seven lowest val MAEs are all `moe` with 16 experts,
the lowest 0.2037 at hidden 64, and the first other mechanism is a `sparse_moe` at 0.2081.
The best trial of each mechanism lies between 0.2037 (`moe`) and 0.2186 (`plain`), and every
one of them is below the baseline of 4.1 (0.2281 ± 0.0023).

Three things stop this table from being read as a ranking of mechanisms.

- **The mechanisms got unequal numbers of trials**: 19 for `moe`, 7 for `sparse_moe`, 4 for
  `highway`, 3 for `deeper`, 2 for `plain` and 1 for `gate`. The best of the ten random
  starting trials was trial 6, a `moe` (0.2043), and TPE then put more than half of the search
  on `moe`. The mechanism with the most draws has the most chances at a low minimum, and the
  "best" of `gate` is the one trial it got.
- **Each trial is one seed.** Take the baseline's seed spread as the spread of one run: the
  difference between two single trials then has a standard error of 0.0032 (4.4 prints it).
  The best trials of `sparse_moe` (0.2081) and `highway` (0.2101) are 0.0020 apart, beside
  that standard error of 0.0032: another seed could put them in the other order.
- **The baseline is not the right control.** It is `plain` at hidden 64 and a constant `lr`
  of 1e-3, while the search also tunes width and `lr`: `plain`'s own best trial (0.2186) is
  0.0095 below the baseline. Part of every gain over the baseline is tuning, not mechanism;
  the distance to `plain` at its own best, and to `deeper`, is what a mechanism may claim.

`sparse_moe`'s trade-off, made concrete: its best trial has 16 experts of width 32 at hidden
128 and trains 191,264 parameters but uses 65,984 (34%) on any one input. With `k=1` one
expert of the 16 works on a window; the rest of the 65,984 are the layers every window passes
(the two outer `Linear`s and the router).

### The score is not the only axis

"The best trial" is half a question: best at what cost? A model is trained to be used, and a
use comes with a bound: the weights must fit in the memory of the device that runs it, or
the forecast must be ready in time. Score the same trials on two measures, val MAE and a
cost, and keep only the trials that no other trial beats. That set is the
**Pareto front**: the models for which no other measured model is at least as good on all
chosen measures and better on one. A trial off the front has another trial that is at
least as good on both measures and better on one, so under any bound on the cost there is a
model at least as good as it.

The plots below are drawn so that up is better on the score: val MAE is better when lower,
so its axis is turned over. On the plot against speed, more is better on both axes (faster
to the right, better score up), and the front is the staircase along the upper-right edge of
the cloud. On the plot against parameters fewer is better, so there the good corner is the
upper left and the front runs along the upper-left edge. Only the corners of a staircase are
models; a flat step says "up to the next model, no trial scores better than this one".

Two costs here:

- **parameters trained**, which the trials already reported. In float32 a parameter takes 4
  bytes, so a parameter bound is a memory bound on the weights.
- **speed**: how many windows a model forecasts per second. What is measured is the latency,
  the wall time of one forward pass over all 33,620 validation windows at once, on this
  machine; the speed is the number of windows divided by that time. The work of a forward
  pass is fixed by the configuration, not by the values of the trained weights, so a rebuilt,
  untrained model of the same configuration takes the same time and nothing has to be
  retrained to measure it. One mechanism is the exception: `sparse_moe` sends every window to
  one expert, so the number of multiplications is fixed too, but it calls each expert
  separately on the windows routed to it and skips an expert that gets none. How many expert
  calls run, and so its time, depends on how many experts the trained router uses. Every finished `sparse_moe`
  trial therefore timed its trained model at the end of training (`trained_ms` in 4.3), and
  the speed front uses that time; every other mechanism is timed rebuilt. The confirmation
  below times the trained models too, to check the claim.

A time is a measurement, not a count: it depends on the processor, on the number of threads
and on whatever else the machine is doing, which is why each time is the median of
`LATENCY_REPEATS` passes. On your machine the speeds will differ, and the speed bound below
is therefore stated relative to the baseline model measured on the same machine.

The plots show two fronts of two measures each, one per cost. A front over all three
measures at once (score, number of weights and speed) keeps every point of the two fronts,
apart from exact ties, and can add more: a model that is neither the smallest nor the fastest
at its score, but small *and* fast enough, is on the three-measure front without being on
either two-measure front. The cell after the choices computes it once, from the same trials.

**Choose under a bound.** Two concrete bounds, each a requirement a model might have to meet
before it is used:

- **at most 50,000 parameters**, 200 KB of float32 weights: a model small enough to run on a
  small controller at each site;
- **at least a third of the baseline's speed**: a model may forecast at most three times
  slower than the `plain` model of 4.1, measured on the same machine.

Under a bound, the choice is the trial with the lowest val MAE among those that meet it.
That trial is always on the front of that cost: a trial that beat it on both measures would
also meet the bound and score lower. So choosing under a bound is walking along the front to
the best corner on the allowed side of the bound. The plots below draw each bound as a dashed vertical line.
Before you run the cell after them: say from the plots which mechanism each bound will pick.

In [ ]:
def pareto_front(points: np.ndarray) -> np.ndarray:
    """Indices of the rows that no other row beats or ties on every column (lower is
    better in each)."""
    keep = []
    for i, p in enumerate(points):
        dominated = np.any(np.all(points <= p, axis=1) & np.any(points < p, axis=1))
        if not dominated:
            keep.append(i)
    return np.array(keep)


def rebuilt(
    mechanism: str, hidden: int, n_experts: int, expert_hidden: int
) -> nn.Module:
    """A fresh model of a configuration, with the initial weights a trial starts from."""
    torch.manual_seed(0)
    return make_model(mechanism, hidden, n_experts, expert_hidden)


CONFIG = ["mechanism", "hidden", "n_experts", "expert_hidden"]
df["latency_new"] = [
    latency_ms(
        rebuilt(
            r["mechanism"],
            int(r["hidden"]),
            int(r["n_experts"]),
            int(r["expert_hidden"]),
        )
    )
    for _, r in df.iterrows()
]
# a dense model takes the same time trained or new; a sparse_moe is timed as trained
df["latency_ms"] = df["trained_ms"].where(
    df["mechanism"] == "sparse_moe", df["latency_new"]
)
df["latency_ms"] = df["latency_ms"].fillna(df["latency_new"]).astype(float)
BASELINE_MS = latency_ms(rebuilt("plain", 64, 4, 64))


def speed(ms: float) -> float:
    """Windows per second: the val windows of one pass divided by its time."""
    return len(xva) / (ms / 1e3)


df["speed"] = df["latency_ms"].map(speed)
BASELINE_SPEED = speed(BASELINE_MS)

PARAM_BOUND = 50_000
SPEED_BOUND = BASELINE_SPEED / 3
# per cost: the axis label, the bound, and the sign that makes lower better for the front
COSTS = {
    "total_params": ("parameters trained (log)", PARAM_BOUND, 1),
    "speed": ("speed: val windows per second (log)", SPEED_BOUND, -1),
}
fronts: dict[str, pd.DataFrame] = {}
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (cost, (label, bound, sign)) in zip(axes, COSTS.items()):
    points = np.column_stack([df["mae"], sign * df[cost]]).astype(float)
    fronts[cost] = df.iloc[pareto_front(points)].sort_values(cost)
    for name, group in df.groupby("mechanism"):
        ax.scatter(group[cost], group["mae"], label=name, s=28, alpha=0.8)
    where = "post" if sign == 1 else "pre"
    ax.step(fronts[cost][cost], fronts[cost]["mae"], where=where, color="k", lw=1.2)
    ax.axvline(bound, color="0.4", ls="--", lw=1)
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.xaxis.set_minor_formatter(plt.NullFormatter())
    ax.set_xlabel(label)
# a lower MAE is better: the axis is turned over, so up is better on both plots
axes[0].invert_yaxis()
axes[0].set_ylabel("val MAE (axis turned over: up is better)")
axes[0].set_title("against parameters: the front is the upper-left edge")
axes[1].set_title("against speed on this machine: the front is the upper-right edge")
axes[0].legend(fontsize=7)
plt.tight_layout()

show = ["mae", *CONFIG, "total_params", "latency_ms", "speed"]
for cost in COSTS:
    print(f"the front on val MAE and {cost}:")
    print(
        shown(fronts[cost][show])
        .round({"latency_ms": 1, "speed": -3})
        .astype({"speed": int})
        .to_string(index=False),
        "\n",
    )
print(
    f"baseline (plain, hidden 64): {BASELINE_MS:.1f} ms per pass over the {len(xva):,} val "
    f"windows, {BASELINE_SPEED:,.0f} windows per second"
)
print(
    f"the bounds (dashed): {PARAM_BOUND:,} parameters, {SPEED_BOUND:,.0f} windows per second"
)
sparse = df[df["mechanism"] == "sparse_moe"]
print(
    "\nsparse_moe trials, ms new against ms trained:",
    ", ".join(
        f"trial {int(r['trial'])} {r['latency_new']:.1f}/{r['latency_ms']:.1f} "
        f"({int(r['experts_used'])} of {int(r['n_experts'])} experts used)"
        for _, r in sparse.iterrows()
    ),
)

In [ ]:
# per bound: the cost, the limit, and the sign that turns "at least" into "at most"
BOUNDS = {
    "params <= 50,000": ("total_params", PARAM_BOUND, 1),
    "speed >= baseline / 3": ("speed", SPEED_BOUND, -1),
}
# a trial is one run; the baseline's seed spread stands in for the spread of one run, so
# the difference of two trials has a standard error of sqrt(2) times it
ONE_RUN_SE = BASELINE.std * np.sqrt(2)

best_trial = df.loc[df["mae"].idxmin()]
chosen: dict[str, pd.Series] = {}
for bound, (cost, limit, sign) in BOUNDS.items():
    inside = df[sign * df[cost] <= sign * limit]
    if inside.empty:
        print(f"{bound}: no trial meets it")
        continue
    pick = inside.loc[inside["mae"].idxmin()]
    chosen[bound] = pick
    gap = pick["mae"] - best_trial["mae"]
    print(
        f"{bound:24s} {len(inside):2d} trials meet it; the choice: trial {int(pick['trial'])}, "
        f"{pick['mechanism']}, hidden {int(pick['hidden'])}, "
        f"{int(pick['total_params']):,} params, {pick['speed']:,.0f} windows/s, "
        f"val MAE {pick['mae']:.4f} ({gap:+.4f} against the best trial); "
        f"on the front: {pick['trial'] in set(fronts[cost]['trial'])}"
    )
print(
    f"\nthe best trial without a bound: trial {int(best_trial['trial'])}, "
    f"{best_trial['mechanism']}, hidden {int(best_trial['hidden'])}, "
    f"{int(best_trial['total_params']):,} params, {best_trial['speed']:,.0f} windows/s, "
    f"val MAE {best_trial['mae']:.4f}"
)
print(f"the standard error of a difference of two single runs: {ONE_RUN_SE:.4f}")

In [ ]:
# the front over all three measures at once: lower MAE, fewer parameters, more speed
points = np.column_stack([df["mae"], df["total_params"], -df["speed"]]).astype(float)
front3 = df.iloc[pareto_front(points)].sort_values("speed", ascending=False)
on_two = set(fronts["total_params"]["trial"]) | set(fronts["speed"]["trial"])
front3 = front3.assign(
    **{
        "on a two-measure front": [
            "yes" if t in on_two else "no" for t in front3["trial"]
        ]
    }
)
print(f"the front on val MAE, parameters and speed: {len(front3)} trials")
print(
    shown(front3[["trial", *show, "on a two-measure front"]])
    .round({"latency_ms": 1, "speed": -3})
    .astype({"speed": int})
    .to_string(index=False)
)

**The front on parameters** has three trials: `highway` at hidden 32 (17,232 parameters, val
MAE 0.2129), `moe` at hidden 32 with 16 experts of width 16 (31,680; 0.2066), and `moe` at
hidden 64 with 16 experts (95,968; 0.2037). **The front on speed** has eight rows: from `plain`
at hidden 64 (11.9 million windows per second; 0.2265) through `deeper` (9.1 million; 0.2127),
a `moe` at hidden 64 with 2 experts (5.1 million; 0.2101), two trained `sparse_moe` at hidden
128 (3.2 and 2.8 million; 0.2098 and 0.2081) and the `moe` at hidden 32 (2.1 million; 0.2066),
up to the `moe` at hidden 64 with 16 experts (1.33 million; 0.2037). Its last two rows (the
top of the plot) are that one configuration trained at two values of `lr`; their speeds differ
only by the noise of the measurement, so one of them is on the front by noise.

**Parameters are not speed.** The `moe` at hidden 32 has 31,680 parameters, fewer than
`deeper` (32,016), and forecasts 2.1 million windows per second against 9.1 million: more than
four times slower. A dense mixture runs all 16 experts on every window, 16 small matrix
products one after another and a router on top. A parameter count says what a model stores;
it does not say how long it computes. The other way round, a trained `sparse_moe` can be
faster than a new one: the four of its seven trials whose trained router left experts idle
were 1.25 to 2.2 times faster after training (trial 29: 27.0 ms new, 12.1 ms trained, its
router using 3 of 16 experts), because the idle experts are skipped. Trial 17, with both of
its 2 experts in use, measured 11.7 ms new and 10.6 ms trained, the same within noise. Trials 3 and 4 measured
slower trained (54.3 against 21.3 ms, 29.1 against 17.8), with the same 3 of 10 experts in use
as trial 11, which was faster trained: their trained time was taken at the end of training
while the machine was busy with other work, and a time taken under load can be off by more
than a factor of two.

**The choices.** Under at most 50,000 parameters, 10 trials qualify, and the choice is trial
16, the `moe` at hidden 32: val MAE 0.2066, 0.0029 above the best trial with a third of its
parameters, beside a standard error of 0.0032 for a difference of two single runs. Under at least a third of
the baseline's speed, 4.0 million windows per second on this machine (the baseline makes 12.0
million), 10 trials qualify, and the choice is trial 26, the `moe` at hidden 64 with 2 experts
of width 32: 5.1 million windows per second, val MAE 0.2101, 0.0064 above the best trial. The best trial of the search meets neither bound. Both choices are on the
front of their cost, as they must be: the same search gives a different model for each bound.
On one seed each, another seed can move each of these gaps by about 0.003; the
confirmation below trains them again.

**The three-measure front** has 10 trials: the nine of the two fronts above and trial 35, a
`moe` at hidden 64 with 8 experts of width 16 (45,400 parameters, 2.5 million windows per
second, val MAE 0.2095). On parameters it loses to trial 16 (fewer parameters, lower MAE), on
speed to trial 29 (faster, lower MAE), but each of those loses to it on the third measure:
trial 16 is slower, trial 29 trains four times its parameters. A bound on both costs at once
that falls between them would pick it: at most 50,000 parameters *and* at least 2.3 million
windows per second. Its speed is 8% above that bound and trial 16's 11% below, closer than a
time taken under load can be trusted, so a bound like this needs the measurement repeated.

### Confirming the winners

The search table compares the *best trial* of each mechanism, and that comparison has three
weaknesses: a mechanism that got more trials has more chances to draw a lucky one, every
trial is a single seed, and the winners were chosen on val, so their val scores are the
maxima of noisy numbers. The confirmation below takes the best trial of each mechanism, and
the two choices under a bound when they are other trials, and trains each configuration
again with five new seeds, 1 to 5 (seed 0 is the seed of the search trial itself, so it
would bring the selected lucky run back in). Every comparison is then made on the
confirmation's val MAE, each difference beside its standard error; the table is sorted on
the val mean. The
rows share their seed numbers but not their runs: models of different shapes draw different
initial weights and batches from the same seed, so nothing pairs run 1 of one row with run 1
of another, and the unpaired standard error of 4.1 applies.
The test windows are read once, after the choices are made, in the cell after the reading.

The routed winners have far more parameters than the best `deeper` trial, so each also gets
a control of its own size: `deeper` with the hidden width at which it has the winner's
parameters, at the `lr` of `deeper`'s best trial. Every row runs at the `lr` of its own
trial, which the table prints: a pair of rows can differ in `lr` as well as in mechanism.

Each row also has two latencies: the rebuilt, untrained model as in the front above (`ms
new`), and the model trained with seed 1 (`ms trained`). If the time of a pass does not
depend on the weights, the two agree up to the noise of the measurement.

In [ ]:
def n_params_of(mechanism: str, hidden: int) -> int:
    return sum(p.numel() for p in make_model(mechanism, hidden).parameters())


# seed 0 is the seed every search trial ran with; the confirmation runs other seeds
CONFIRM_SEEDS = range(1, SEEDS + 1)


def confirm(
    mechanism: str, hidden: int, n_experts: int, expert_hidden: int, lr: float
) -> dict:
    """Train one configuration with the seeds CONFIRM_SEEDS and score it on val and test.
    Returns both as a Spread, its parameters and lr, and the latency of the rebuilt model
    and of the model trained with the first seed."""
    vals, tests = [], []
    for seed in CONFIRM_SEEDS:
        torch.manual_seed(seed)
        model = make_model(mechanism, hidden, n_experts, expert_hidden)
        v, t = run(model, lr=lr)
        vals.append(v)
        tests.append(t)
        if seed == CONFIRM_SEEDS[0]:
            trained_ms = latency_ms(model)
            n_used = experts_used(model) if mechanism == "sparse_moe" else None
    return {
        "val": Spread(vals),
        "test": Spread(tests),
        "params": sum(p.numel() for p in model.parameters()),
        "hidden": hidden,
        "lr": lr,
        "new_ms": latency_ms(rebuilt(mechanism, hidden, n_experts, expert_hidden)),
        "trained_ms": trained_ms,
        "experts_used": n_used,
    }


def confirm_row(row: pd.Series) -> dict:
    return confirm(
        row["mechanism"],
        int(row["hidden"]),
        int(row["n_experts"]),
        int(row["expert_hidden"]),
        float(row["lr"]),
    )


best_per_mechanism = df.loc[df.groupby("mechanism")["mae"].idxmin()].set_index(
    "mechanism", drop=False
)

confirmed: dict[str, dict] = {}
for name, row in best_per_mechanism.iterrows():
    confirmed[name] = confirm_row(row)

# a choice under a bound that is not already the best trial of its mechanism gets a row of its own
chosen_row: dict[str, str] = {}
for bound, pick in chosen.items():
    same = [n for n, r in best_per_mechanism.iterrows() if r["trial"] == pick["trial"]]
    label = same[0] if same else f"{pick['mechanism']} h{int(pick['hidden'])}"
    if not same:
        confirmed[label] = confirm_row(pick)
    chosen_row[bound] = label

# the control of a routed winner: `deeper` as wide as it takes to match its parameters
for routed in ROUTED:
    if routed not in confirmed or "deeper" not in best_per_mechanism.index:
        continue
    target = confirmed[routed]["params"]
    width = min(range(16, 512), key=lambda h: abs(n_params_of("deeper", h) - target))
    confirmed[f"deeper ~ {routed}"] = confirm(
        "deeper", width, 4, width, float(best_per_mechanism.loc["deeper", "lr"])
    )

ranked = sorted(confirmed.items(), key=lambda kv: kv[1]["val"].mean)
top_name, top = ranked[0]


def against_top(spread: Spread) -> str:
    """The difference with the top row's val mean, and its standard error."""
    diff = spread.mean - top["val"].mean
    return f"{diff:>+9.4f} {se_of_difference(spread, top['val']):>7.4f}"


print(
    f"{'':20s} {'hidden':>6} {'params':>9} {'lr':>8} {'val MAE':>17} "
    f"{f'- {top_name}':>9} {'se':>7} {'ms new':>7} {'ms trained':>10}"
)
print("-" * 102)
print(
    f"{'baseline':<20} {64:>6} {n_params_of('plain', 64):>9,} {1e-3:>8.1e} "
    f"{BASELINE!s:>17} {against_top(BASELINE)} {BASELINE_MS:>7.1f}"
)
for name, r in ranked:
    vs_top = against_top(r["val"]) if name != top_name else f"{'':>17}"
    print(
        f"{name:<20} {r['hidden']:>6} {r['params']:>9,} {r['lr']:>8.1e} {r['val']!s:>17} "
        f"{vs_top} {r['new_ms']:>7.1f} {r['trained_ms']:>10.1f}"
    )

for name, r in confirmed.items():
    if r["experts_used"] is not None:
        print(
            f"{name}: trained with seed {CONFIRM_SEEDS[0]}, its router sends the val windows "
            f"to {r['experts_used']} experts"
        )
print(f"the speed bound in ms: {len(xva) / SPEED_BOUND * 1e3:.2f}")
print("\npairs on val, the difference and its standard error:")
for a, b in dict.fromkeys(
    [
        ("moe", "highway"),
        ("moe", "deeper ~ moe"),
        ("gate", "deeper"),
        ("sparse_moe", "deeper ~ sparse_moe"),
        ("plain", "baseline"),
        *[(label, "moe") for label in chosen_row.values() if label != "moe"],
        *[(label, "deeper") for label in chosen_row.values() if label != "deeper"],
    ]
):
    ra = BASELINE if a == "baseline" else confirmed.get(a, {}).get("val")
    rb = BASELINE if b == "baseline" else confirmed.get(b, {}).get("val")
    if ra is not None and rb is not None and a != b:
        print(
            f"  {a} - {b}: {ra.mean - rb.mean:+.4f}, se {se_of_difference(ra, rb):.4f}"
        )

Every reading below is on val, each difference beside its standard error over the five new
seeds (the columns `- moe` and `se`, and the pairs under the table).

- **`moe` stays in front.** 0.2066 ± 0.0025, ahead of `highway` (0.2128 ± 0.0017) by 0.0062
  (standard error 0.0014), and of `sparse_moe` (0.2139 ± 0.0032) by 0.0073 (0.0018), which
  trains twice its parameters. Its search trial scored 0.2037; over new seeds the same configuration
  scores 0.0029 more. That is the maximum of noisy numbers from the introduction above: the
  trial was chosen because its one seed came out low.
- **`moe` against a model of its own size: a small lead.** `deeper ~ moe` is `deeper` at
  hidden 161 (95,939 parameters): 0.2092 ± 0.0014 against `moe`'s 0.2066 ± 0.0025, a gap of
  0.0027, standard error 0.0013. And `deeper ~ sparse_moe`, the same block at hidden 271 with
  191,199 parameters, reaches 0.2078 ± 0.0028: 0.0012 behind `moe` (standard error 0.0017),
  and it is faster (15.9 ms trained against 24.0). So routing leads a plain block of its own
  size by 0.0027, and a plain block twice as large by 0.0012, a gap that another set of seeds
  could close or reverse. Both controls run untuned, at
  the `lr` of `deeper`'s best trial (2.0e-4 against `moe`'s 2.9e-4); E3 tunes one.
- **`gate` and its capacity-matched control end level.** `gate` and `deeper` both have hidden
  64 and 32,016 parameters: 0.2145 ± 0.0011 against 0.2142 ± 0.0014, a gap of 0.0003 (standard
  error 0.0008). The pair differs in `lr` as well (1.1e-4 for `gate`'s one trial, 2.0e-4 for
  `deeper`'s best), so even a larger gap would be one between two configurations as the
  search drew them, not a clean one-factor control.
- **`sparse_moe` is worse than a dense block of its size.** 0.2139 ± 0.0032 against 0.2078 ±
  0.0028 for `deeper ~ sparse_moe`, 0.0061 behind (standard error 0.0019), while it uses
  65,984 parameters per window.
- **Part of the gain over the baseline is tuning.** `plain` at hidden 128 with its own `lr`
  reaches 0.2176 ± 0.0021 against the baseline's 0.2281 ± 0.0023, 0.0105 lower (standard
  error 0.0014).

The two choices under a bound:

- **Under the parameter bound, the choice is close behind the best.** `moe h32` scores
  0.2090 ± 0.0012 against 0.2066 ± 0.0025 for the best `moe`: a gap of 0.0025 (standard error
  0.0012). With a third of the parameters it is slightly worse than the best model, and
  0.0052 ahead of `deeper` at the same size (standard error 0.0008).
- **Under the speed bound, the choice and a faster model end close.** The bound is
  8.40 ms per pass here. `moe h64` (trial 26) runs at 5.9 ms trained and scores 0.2134 ±
  0.0016, the lowest val mean of the confirmed models inside the bound; `deeper` (0.2142 ±
  0.0014), `gate` (0.2145) and `plain` (0.2176) follow. `deeper` is 0.0007 higher (standard
  error 0.0010), and at 3.6 ms it is 1.6 times faster. `moe h64` is 0.0068 behind the best
  `moe` (standard error 0.0013). `deeper ~ moe` would be the stronger choice on score (0.2092), but trained it took
  8.9 ms, just outside the bound, in a measurement taken while the machine was busy; a repeat
  can land on either side of 8.40.
- **The speed of a dense model does not depend on its weights; that of `sparse_moe` does.**
  For every dense model `ms new` and `ms trained` agree to within about a millisecond (`moe`
  24.3 against 24.0, `deeper ~ moe` 9.2 against 8.9, `gate` 3.8 against 3.9). `sparse_moe`
  takes 22.4 ms new and 11.1 ms trained: its router trained with seed 1 sends the val windows
  to 3 of its 16 experts, as the router of its search trial did. Another seed trains another
  router, which may use another number of experts. The same model measured at two moments
  also differs: the best `moe` took 25.2 ms on the front and 24.3 ms here. A bound close to a
  model's time needs the measurement repeated, and for `sparse_moe` repeated over seeds.

So the search proposed, and the seeds and the controls checked it on val: `moe` leads, by
0.0027 over a plain block of its own size and by 0.0012 over one twice as large; `gate` ends
level with its control; `sparse_moe` is 0.0061 behind its; the choice under the parameter
bound is 0.0025 behind the best model with a third of its parameters, and the choice under
the speed bound is the lowest val MAE inside the bound, 0.0007 ahead of the faster `deeper`. The next cell reads the test windows
once, for the models the rule of lowest val MAE chose.

In [ ]:
# the choice on the confirmation's val column, among the confirmed models that meet
# each bound; the speed of a model is that of its trained model
for r in confirmed.values():
    r["speed"] = speed(r["trained_ms"])
final: dict[str, str] = {}
for bound, ok in {
    "no bound": lambda r: True,
    "params <= 50,000": lambda r: r["params"] <= PARAM_BOUND,
    "speed >= baseline / 3": lambda r: r["speed"] >= SPEED_BOUND,
}.items():
    inside = {n: r for n, r in confirmed.items() if ok(r)}
    if inside:
        final[bound] = min(inside, key=lambda n: inside[n]["val"].mean)

print("the test windows, once, for the models chosen on val:")
print(f"  {'baseline (4.1)':<38} test {BASELINE_TEST}")
for bound, name in final.items():
    r = confirmed[name]
    print(
        f"  {bound + ': ' + name:<38} test {r['test']}   "
        f"({r['params']:,} params, {r['speed']:,.0f} windows/s)"
    )

The test windows are a later stretch of each series and every model scores higher on them,
the baseline included (0.2607 ± 0.0016 against 0.2281 on val). All three choices are below
the baseline: the best `moe` 0.2386 ± 0.0023, the `moe` under the parameter bound 0.2406 ±
0.0024 and the `moe` with 2 experts under the speed bound 0.2475 ± 0.0020.
This is the report, not another comparison: nothing is chosen on these numbers. One data
point is not a law either: this is one series with one set of 20 sites.

## 4.5 Stopping the hopeless trials early

Every trial above ran its 15 epochs, including the ones that were clearly behind after
three. A **pruner** reads the partial curves the trials report and stops a trial that is
behind, so its remaining epochs go to other trials.

**A pruner is not a scheduler.** Both spend effort coarse first and fine after, but they act
on different things:

| | learning-rate scheduler (4.2) | pruner |
|---|---|---|
| acts on | the `lr` of the optimizer, inside one training | the trials of a search |
| decides | how large the next updates of the weights are | whether a trial gets more epochs at all |
| reads | the epoch count (`T_max` for the cosine) | the val MAE every trial reports |
| what a trial under it does | trains all its epochs, with smaller steps later | runs on, or stops, and its remaining epochs are never trained |

A scheduler cannot save a single epoch of a search; a pruner does not change one update of
a trial that it lets continue. The search of 4.3 has no scheduler (every trial trains at a
constant `lr`), and this section adds a pruner to it.

**Successive halving** lets the trials compete for training budget in rungs. Many trials
start with a small budget; at each rung only the best fraction goes on to a larger budget,
the rest are stopped. In its original form the trials of a rung are compared all at once,
which means waiting until every trial has reached it. **ASHA**, asynchronous successive
halving, decides as each trial arrives: a trial that reaches a rung is compared with the
trials that reached that rung before it. Optuna's `SuccessiveHalvingPruner` is the
asynchronous version. Here the rungs are at 3 and 9 epochs (`min_resource=3`, times
`reduction_factor=3`); a trial that survives both runs its 15 epochs.

**The rule at a rung, exactly.** Count the trials that have reached the rung, this one
included: n. The trial goes on only if it is among the best `max(n // 3, 1)` of them. For n of
3 or more that is the best third of the trials at the rung so far, rounded down (at n = 5 one
trial goes on); for the first two arrivals it is more lenient, since the one in the lead goes
on. What the arrival order
changes is not the fraction but *whom* a trial is compared with: the first trials meet a
small, arbitrary handful, the later ones a pool that the sampler has already shaped. A trial
in the best third of all 36 can still be stopped if the trials before it happened to start
fast.

**When stopping early saves budget.** A pruner bets that a trial behind at a rung would also
be behind at the end. When the ranking after 3 epochs predicts the ranking after 15, it saves
the epochs of the losers and loses nothing. A trial can drop out wrongly when it learns late
(a slow start and a strong finish), when the rung is short enough that the noise of one run
decides, or when the trials it is compared with are fast starters that fall back later.

**What the sampler does with a stopped trial.** TPE's good pile (4.3) is filled with finished
trials first, ordered by their score; a stopped trial enters it only when there are not
enough finished ones. So every finished trial is preferred to every stopped one, whatever
score the stopped one had when it stopped.

Two checks. First the premise, on the trials of 4.3: does the ranking after 3 epochs predict
the ranking after 15? It is measured with the **Spearman rank correlation**: rank the trials
by their val MAE after 3 epochs and again after 15, and correlate the two rankings. It is 1
when the order is the same, 0 when the two orders are unrelated, and it looks only at the
order, not at the size of the differences. Second, the same search with the pruner switched
on, as the study `lesson3-search-pruned`: the same space and the same TPE with the same seed.
The first 10 trials are TPE's random start, which does not look at any result, so they are
the same configurations in both searches; from trial 10 on TPE proposes from the results,
and the two searches diverge. The cell below shows those first ten trials as the pruner will
meet them, in order, after 3 epochs.

In [ ]:
GRACE = min(3, EPOCHS)  # the first epoch after which the pruner may stop a trial

# the first ten trials of the full search, in the order they ran, at the first rung; the
# pruned search starts with the same ten configurations
first_ten = pd.DataFrame(
    [
        {
            "trial": t.number,
            "mechanism": t.user_attrs["config"]["mechanism"],
            "hidden": t.user_attrs["config"]["hidden"],
            f"val MAE after {GRACE} epochs": round(-t.intermediate_values[GRACE], 4),
        }
        for t in full.study.trials[:10]
    ]
)
print(first_ten.to_string(index=False))

**Predict before the numbers come.** Apply the rule to this table, in order: which of the
first ten trials does the pruner stop after 3 epochs? (Trial 0 meets no one; trial 6 meets
trials 0 to 5.) Which mechanisms do you expect to be behind after 3 epochs over the whole
search, will the configuration that wins the full search survive the first rung, and will the
pruned search end with a best trial as good as the full search's?

In [ ]:
RUNGS = [r for r in (GRACE, 3 * GRACE) if r < EPOCHS]

# a trial reports minus the MAE at step = epochs trained; the minus is undone here
curve_of = {
    t: pd.Series(full.study.trials[t].intermediate_values).mul(-1) for t in df["trial"]
}
at_grace = pd.Series({t: c[GRACE] for t, c in curve_of.items()})
at_end = pd.Series({t: c[EPOCHS] for t, c in curve_of.items()})
rank_corr = at_grace.corr(at_end, method="spearman")
winner = int(at_end.idxmin())
winner_rank = int(at_grace.rank().loc[winner])
print(
    f"rank correlation (Spearman) of val MAE after {GRACE} epochs and after {EPOCHS}: "
    f"{rank_corr:.2f}"
)
print(
    f"the final winner (trial {winner}, {df.set_index('trial').loc[winner, 'mechanism']}) "
    f"was number {winner_rank} of {len(at_grace)} after {GRACE} epochs "
    f"(the best third: {len(at_grace) // 3})"
)

mechanism_of = df.set_index("trial")["mechanism"]
for name in ("moe", "highway"):
    of = mechanism_of[mechanism_of == name].index
    if len(of) > 2:
        within = at_grace[of].corr(at_end[of], method="spearman")
        print(f"the same, within the {len(of)} {name} trials: {within:.2f}")
ranks = pd.DataFrame(
    {
        f"mean rank after {GRACE}": at_grace.rank().groupby(mechanism_of).mean(),
        f"mean rank after {EPOCHS}": at_end.rank().groupby(mechanism_of).mean(),
        "trials": mechanism_of.value_counts(),
    }
)
print(f"\nrank 1 is the lowest val MAE, of {len(at_grace)} trials")
print(ranks.sort_values(f"mean rank after {EPOCHS}").round(1).to_string())

fig, ax = plt.subplots(figsize=(8, 4))
colour = {m: f"C{i}" for i, m in enumerate(sorted(df["mechanism"].unique()))}
for t, c in curve_of.items():
    ax.plot(c.index, c.values, color=colour[mechanism_of[t]], lw=1, alpha=0.7)
for m, col in colour.items():
    ax.plot([], [], color=col, label=m)
for r in RUNGS:
    ax.axvline(r, color="k", ls="--", lw=0.8)
ax.set_xlabel("epoch")
ax.set_ylabel("val MAE")
ax.set_ylim(top=1.05 * float(at_grace.max()))
ax.set_title(
    "every trial of the full search; dashed: the rungs where the pruner decides"
)
ax.legend(fontsize=7)
plt.tight_layout()

In [ ]:
def spearman_interval(r: float, n: int) -> tuple[float, float]:
    """An approximate 95% interval for a Spearman correlation r over n trials: Fisher's
    z = artanh(r) is close to normal with variance 1.06 / (n - 3)."""
    if n <= 3:
        return float("nan"), float("nan")
    z, se = np.arctanh(np.clip(r, -0.999, 0.999)), np.sqrt(1.06 / (n - 3))
    return float(np.tanh(z - 1.96 * se)), float(np.tanh(z + 1.96 * se))


lo, hi = spearman_interval(rank_corr, len(at_grace))
print(f"all {len(at_grace)} trials: {rank_corr:.2f}, 95% interval {lo:.2f} to {hi:.2f}")
for name in ("moe", "highway"):
    of = mechanism_of[mechanism_of == name].index
    if len(of) > 3:
        r = at_grace[of].corr(at_end[of], method="spearman")
        lo, hi = spearman_interval(r, len(of))
        print(
            f"within the {len(of):2d} {name:7s} trials: {r:.2f}, 95% interval {lo:.2f} to {hi:.2f}"
        )
        if len(of) <= 8:
            orders = itertools.permutations(range(len(of)))
            rhos = [
                pd.Series(range(len(of))).corr(pd.Series(o), method="spearman")
                for o in orders
            ]
            share = np.mean(np.array(rhos) >= r - 1e-9)
            print(
                f"  share of random orders of {len(of)} trials that reach {r:.2f}: {share:.2f}"
            )

# the moe trials of one expert layout, 16 experts of width 32; they differ in hidden and lr
layout = [
    t
    for t in at_end.index
    if full.study.trials[t].user_attrs["config"]["mechanism"] == "moe"
    and full.study.trials[t].user_attrs["config"]["n_experts"] == 16
    and full.study.trials[t].user_attrs["config"]["expert_hidden"] == 32
]
if layout:
    same = pd.DataFrame(
        {
            "hidden": [
                full.study.trials[t].user_attrs["config"]["hidden"] for t in layout
            ],
            "lr": [full.study.trials[t].user_attrs["config"]["lr"] for t in layout],
            f"val MAE after {GRACE}": at_grace[layout].round(4).values,
            f"after {EPOCHS}": at_end[layout].round(4).values,
        },
        index=pd.Index(layout, name="trial"),
    ).sort_values(f"after {EPOCHS}")
    print(f"\nthe {len(layout)} moe trials with 16 experts of width 32:")
    print(same.to_string(float_format=lambda v: f"{v:.4g}"))

**Read against your prediction.** Over all 36 trials the rank correlation is 0.36. Like a
mean, a correlation over a sample moves by chance, and its approximate 95% interval (the cell
above) runs from 0.03 to 0.62: the order after 3 epochs is a weak guide to the order after 15,
and the 36 trials only just show that it is a guide at all.

Within one mechanism the numbers are too few and too close to read. The 19 `moe` trials give
0.14 (interval −0.34 to 0.57), the 4 `highway` trials 0.80 (−0.73 to 1.00), and 1 in 6 random
orders of four trials reaches 0.80: the two intervals overlap almost completely. And eight of
the `moe` trials share one expert layout, 16 experts of width 32; six of them finished within
0.0013 of each other (0.2037 to 0.2050), less than half the standard error of a difference of two
single runs (0.0032, 4.4). No rung can predict the order of values that close, for any mechanism. Neither number
says how a mechanism learns.

The trial that finished first, a `moe` (trial 20), was 11th of 36 after 3 epochs, inside the
best third (12). The table of the first ten shows what a weak correlation does to one trial:
trial 6, a `moe`, was 4th of the ten after 3 epochs, outside the best three, and the best of
them after 15 (0.2043).

The mean ranks per mechanism, in this sample, do not show `moe` starting slowly. The sample is
what TPE chose to draw, so the 19 `moe` trials were drawn where TPE's good pile pointed, not
spread evenly over `moe`. After 3 epochs `highway` (14.0), `deeper` (14.7) and `moe` (14.9)
are level. After 15, `moe` has moved up to 12.9 while `highway` fell back to 21.2 and `deeper`
to 30.0. `sparse_moe` is the one mechanism that starts behind and climbs, from 27.0 to 21.0.
In the plot the red `moe` curves are spread through the pack at the first dashed line and
gather at the bottom by epoch 15.

So a rung at epoch 3 judges on a weak signal, and which trials go on also depends on whom a
trial meets there. With this in hand, predict once more before the next cell: will the pruned
search train a `moe` to the end?

In [ ]:
pruner = optuna.pruners.SuccessiveHalvingPruner(min_resource=GRACE, reduction_factor=3)
pruned, pruned_seconds = search("lesson3-search-pruned", pruner=pruner)
df_pruned = trials_frame(pruned)

finished = df_pruned[~df_pruned["pruned"]]
print(f"{'':16s} {'no pruner':>10} {'pruner':>10}")
print(f"{'seconds':16s} {full_seconds:>10.0f} {pruned_seconds:>10.0f}")
print(
    f"{'epochs trained':16s} {int(df['epochs'].sum()):>10d} {int(df_pruned['epochs'].sum()):>10d}"
)
print(
    f"{'trials stopped':16s} {int(df['pruned'].sum()):>10d} {int(df_pruned['pruned'].sum()):>10d}"
    f"   (of {len(df_pruned)})"
)
print(
    "epochs a trial of the pruned search ran:",
    df_pruned["epochs"].value_counts().sort_index().to_dict(),
)
if finished.empty:
    print("no trial of the pruned search ran all its epochs")
else:
    top = finished.loc[finished["mae"].idxmin()]
    print(f"{'best val MAE':16s} {df['mae'].min():>10.4f} {top['mae']:>10.4f}")
    print(
        f"best trial of the pruned search: trial {int(top['trial'])}, {top['mechanism']}, "
        f"hidden={int(top['hidden'])}, lr={top['lr']:.1e}, {int(top['epochs'])} epochs"
    )
    print(
        "mechanisms of the trials that ran all epochs:",
        finished["mechanism"].value_counts().to_dict(),
    )


def rung_table(rung: int, first: int = 10) -> pd.DataFrame:
    """The pruner's decision at `rung` for the first trials of the pruned search: how many
    trials had reached the rung, counting this one (n), how many of them go on
    (max(n // 3, 1)), and where this trial ranked; next to it what the same configuration
    reached in the full search, where the first 10 trials are the same random draws."""
    trials = pruned.study.trials
    rows = []
    for t in trials[:first]:
        if rung not in t.intermediate_values:
            continue
        at_rung = [
            -u.intermediate_values[rung]
            for u in trials[: t.number + 1]
            if rung in u.intermediate_values
        ]
        mine = -t.intermediate_values[rung]
        rows.append(
            {
                "trial": t.number,
                "mechanism": t.user_attrs["config"]["mechanism"],
                f"val MAE at {rung}": round(mine, 4),
                "n": len(at_rung),
                "go on": max(len(at_rung) // 3, 1),
                "rank": 1 + sum(v < mine for v in at_rung),
                "pruned search": f"{t.state.name.lower()}, {t.user_attrs['epochs_trained']} ep",
                f"full search, epoch {EPOCHS}": round(
                    -full.study.trials[t.number].intermediate_values[EPOCHS], 4
                ),
            }
        )
    return pd.DataFrame(rows)


for rung in [GRACE, 3 * GRACE]:
    table = rung_table(rung)
    if len(table) and rung < EPOCHS:
        print(f"\nthe rung at epoch {rung}, the first 10 trials:")
        print(table.to_string(index=False))


def good_pile(log: StudyLog, before: int) -> list[str]:
    """The trials in TPE's good pile when it proposes trial `before`, as Optuna's TPE forms
    it: ceil(0.1 n) of the n trials so far (Optuna's default gamma), filled with finished
    trials by value first, then stopped ones by how far they got and their value."""
    trials = log.study.trials[:before]
    size = int(np.ceil(0.1 * len(trials)))
    done = sorted(
        (t for t in trials if t.state == TrialState.COMPLETE), key=lambda t: -t.value
    )
    stopped = sorted(
        (t for t in trials if t.state == TrialState.PRUNED),
        key=lambda t: (
            -max(t.intermediate_values),
            -t.intermediate_values[max(t.intermediate_values)],
        ),
    )
    pile = (done + stopped)[:size]
    return [f"{t.number} {t.user_attrs['config']['mechanism']}" for t in pile]


print("\nTPE's good pile when it proposes a trial:")
for log in (full, pruned):
    print(f"  {log.name}:")
    for before in (10, 12, 15, 20, 30):
        if before < len(log.study.trials):
            print(f"    trial {before:2d}: {', '.join(good_pile(log, before))}")

In [ ]:
# the highway trials at hidden 128 that the pruned search trained to the end
h128 = finished[(finished["mechanism"] == "highway") & (finished["hidden"] == 128)]
if len(h128):
    print(
        f"{len(h128)} highway trials at hidden 128 ran all {EPOCHS} epochs in the pruned "
        f"search: lr {h128['lr'].min():.2e} to {h128['lr'].max():.2e}, "
        f"val MAE {h128['mae'].min():.4f} to {h128['mae'].max():.4f}"
    )


# seconds per epoch of the first ten trials, the same configurations in both searches
def per_epoch(log: StudyLog, t: int) -> float:
    attrs = log.study.trials[t].user_attrs
    return attrs["train_seconds"] / attrs["epochs_trained"]


first = range(min(10, len(full.study.trials), len(pruned.study.trials)))
print(f"\n{'trial':>5} {'mechanism':>10} {'s/epoch, full':>14} {'s/epoch, pruned':>16}")
for t in first:
    mechanism = full.study.trials[t].user_attrs["config"]["mechanism"]
    print(
        f"{t:5d} {mechanism:>10} {per_epoch(full, t):14.1f} {per_epoch(pruned, t):16.1f}"
    )
train_full = sum(t.user_attrs["train_seconds"] for t in full.study.trials)
train_pruned = sum(t.user_attrs["train_seconds"] for t in pruned.study.trials)
at_pruned_rates = train_full + sum(
    EPOCHS * per_epoch(pruned, t) - full.study.trials[t].user_attrs["train_seconds"]
    for t in first
)
print(
    f"\ntraining seconds: full {train_full:.0f}, pruned {train_pruned:.0f} "
    f"({train_pruned / train_full:.0%}); with the first ten trials of the full search at "
    f"the pruned search's rates: full {at_pruned_rates:.0f}, pruned "
    f"{train_pruned / at_pruned_rates:.0%}"
)

**What the pruner bought, and what it cost.** It stopped 28 of 36 trials (22 after epoch 3,
6 after epoch 9) and trained 240 epochs instead of 540 (44%). The best trial it found, trial
19, a `highway` with hidden 128 at `lr` 2.0e-4, reaches 0.2090 against the full search's
0.2037: a gap of 0.0053, beside a standard error of 0.0032 for a difference of two single runs.
The confirmation of 4.4 points the same way: the `highway` at hidden 128 confirmed there, at
the `lr` of the full search's best `highway` (2.8e-4), is 0.0062 behind `moe` on val
(standard error 0.0014). Trial 19's own configuration was not confirmed, but the store
speaks for it: the six `highway` trials at hidden 128 that the pruned search ran to the end,
at `lr` 1.97e-4 to 2.75e-4, all ended between 0.2090 and 0.2101 (the cell above). In that range
`lr` barely moves this configuration, so the confirmed one at 2.8e-4 stands in for trial 19. In
this run stopping early most likely cost the search its best model.

**Why, from the rung tables.** The first ten trials are the same random draws in both
searches, so the last column shows where each would have ended without the pruner.

- **Every stopped trial was outside the best third of the trials it met.** Of the first ten,
  six were stopped at epoch 3 (trials 2, 3, 4, 6, 7, 9) and one at epoch 9 (trial 8). Trials 2
  and 3 were last of 3 and of 4, trial 4 was 3rd of 5. The rule is lenient only at the very
  start: trial 0 went on as the first arrival, although it was 5th of the first ten after 3
  epochs (0.2184).
- **Trial 6 is a wrong drop.** The `moe` with hidden 128 and 16 experts was 3rd of the 7
  trials at epoch 3 (0.2171), when 2 go on, behind trial 5 (`highway`, 0.2128) and trial 1
  (`deeper`, 0.2160). Its arrival is not what stopped it: among all of the first ten it was
  4th after 3 epochs, still outside the best three, and the best after 15 (0.2043). That is
  what a correlation of 0.36 between the two orders allows for one trial.
- **Trial 8, at epoch 9,** met a set of four: a `highway` 2nd of 4, when only the best goes
  on. It would have ended at 0.2115, behind trial 5 (0.2101), which went on.
- **Trial 4 is a second wrong drop, a late learner.** The `sparse_moe` at hidden 64 was 3rd of 5 at epoch 3
  (0.2218) and stopped; without the pruner it ended at 0.2096, the second best of the ten
  after trial 6. Trial 3, also a `sparse_moe`, went from 0.2290 to 0.2174.

**What TPE made of it.** The good pile is the best tenth of the trials so far (4.3): one trial
when TPE proposes trial 10, two up to trial 20, three or four after. In the full search that
one trial was trial 6, a `moe`, and from then on the pile held `moe` trials (the printout
above): that is how the full search came to spend 19 of its 36 trials on `moe`. In the pruned
search the finished trials at trial 10 were 0 and 1 (`deeper`) and 5 (`highway`); finished
trials go into the pile before stopped ones, so the pile of one was trial 5, a `highway`, and
from trial 15 on it held only `highway` trials. TPE proposed from them: of the 8 trials that
ran all 15 epochs, 6 are `highway` and 2 `deeper`, none routed. `highway` starts fast, so the
trials a later arrival met at epoch 3
were fast starters: 11 of the 12 best epoch-3 scores of the pruned search are `highway`, and
trial 6's 0.2171 would rank 14th among them. Both searches drifted toward the mechanism of
their first good pile; the pruner decided which mechanism that was.

**Seconds.** 3443 s for the full search against 465 s with the pruner, 14% of the seconds for
44% of the epochs. Most of that gap is the machine, not the pruner: the first ten trials of
the full search ran while the machine was busy with other work, and trials 3 to 7 took about
three to ten times as long per epoch as the same configurations in the pruned search (the
table above). At the pruned search's rates for those ten, the full search would have trained
about 2,020 s, and the pruned search takes 23% of that. The seconds still fall more than the
epochs (23% against 44%) because the two searches trained different models: the full search
spent 19 trials on `moe` with up to 16 experts, which takes longer per epoch than a `highway`
(7.1 s against 2.0 s for trials 6 and 5 of the pruned search). Wall seconds count whatever
else the machine is doing, so the epochs are the cost that carries over to another machine.

**The budget question.** The 300 epochs that the pruner saved would pay for 20 more full
trials, or for about 45 more at the pruned search's average of under 7 epochs per trial. For a
fixed budget the choice is therefore between 36 full trials and more than twice as many, most
of them stopped after three epochs. Which wins depends on how well three epochs predict
fifteen: here weakly (0.36 over all 36 trials, interval 0.03 to 0.62), and on whom the trials meet
at a rung, which the sampler shapes (E6).

## 4.6 Where the searches are

Both searches are studies in the course's Optuna store, the one every search of this course
writes to: `lesson3-search` with every trial run to the end, `lesson3-search-pruned` with
the pruner. Every trial carries what it drew (`config`), its val MAE after the last epoch it
ran (`val_mae`), how many epochs that was (`epochs_trained`), its parameters trained and used,
and its curve per epoch. `make optuna-dashboard` opens the store: the history plot of a study
shows the trials in order, and the intermediate-values plot shows the curve of every trial,
the stopped ones ending at epoch 3 or 9.

In [ ]:
for log in (full, pruned):
    states = pd.Series([t.state.name for t in log.trials()]).value_counts().to_dict()
    print(f"{log.name:22s} {len(log.trials()):>3} trials in the store  {states}")
print(f"\nthe store: {full.store.db}")

## What to take from this

- **Name the baseline first, and the right one.** 4.1 fixed the number to beat before any
  trial ran. The search then went below it with every mechanism, `plain` included, because it
  also tunes width and `lr`: a search result is read against a control that had the same
  tuning.
- **A scheduler changes the learning rate within one training.** Read from the optimizer,
  `cosine 3e-3` trains at 3.00e-3 in epoch 1, 1.66e-3 in epoch 8 and 3.3e-5 in epoch 15. Under
  Adam an update is on average the `lr` times a factor below 1 (here 0.13 to 0.18, about a
  sixth), so within one training a smaller `lr` gives a proportionally shorter update. `plain` with
  cosine decay from 1e-3 scores 0.2165 against 0.2281 at a constant 1e-3, 0.0117 lower
  (standard error 0.0010) (4.2).
- **A pruner stops whole trials; it leaves the `lr` alone.** A scheduler saves no epoch of a
  search, and a pruner changes no update of a trial it lets continue (4.5).
- **The search proposes; seeds and controls check it, on val.** The best trial of a search
  is the minimum of noisy numbers, drawn unequally often from each mechanism (19 `moe` trials,
  1 `gate`). Retrained over new seeds, `moe` kept its lead over the other mechanisms, by
  0.0027 (standard error 0.0013) over a plain block of its own size and by 0.0012 (0.0017)
  over one twice its size; `gate` ended level with its control, `sparse_moe` 0.0061 behind
  its. Test was read once, for the
  models chosen (4.4).
- **Choose on the Pareto front, under a bound you can name.** The front holds the models no
  other measured model matches on every measure: here score against parameters, score against
  speed, and all three at once, where a `moe` with 8 experts is on the front of three measures
  and on neither front of two. Under at most 50,000 parameters the choice, a `moe` with a third
  of the best one's parameters, is 0.0025 behind it (standard error 0.0012); under at least a
  third of the baseline's speed the choice, a `moe` with 2 experts, ended 0.0007 ahead of a
  faster `deeper` (standard error 0.0010) once trained over new seeds. Parameters are not speed: a 31,680-parameter `moe` is more than four times slower
  than the 32,016-parameter `deeper` (4.4).
- **Successive halving lets trials compete for budget, and an asynchronous pruner judges a
  trial against whoever reached the rung before it.** At a rung only the best `max(n // 3, 1)`
  of the n trials so far go on: the best third, and the leader alone for the first two. The
  pruner trained 44% of the epochs, but 3 epochs predict 15 weakly (0.36 over all trials), and
  it made two wrong drops: the `moe` that would have been the best of the first ten, and the
  late-learning `sparse_moe` of trial 4. TPE, which puts finished trials in its good pile
  before stopped ones, then drew fast-starting `highway` trials, a `moe` met a rung full of
  them, and the search never trained a routed model to the end. Its best trial was 0.0053
  behind on one seed, and a configuration like it confirmed 0.0062 behind `moe` (4.5).
- **Parameters trained and parameters used are different numbers for `sparse_moe`, and the
  same number for everything else.** The best `sparse_moe` trial trains 191,264 and uses 65,984
  per input (4.4).
- **A search space grows exponentially with its axes, a budget by addition.** k axes of three
  values give 3^k points; here 324 discrete combinations and a continuous `lr` against 36
  trials (4.3).

## Exercises

**E1 -- predict before you read 4.4.** Given `01_gates_highways_moe`'s result on an
electricity window (nothing beat a plain layer of the same size), decide what you expect here,
on a series with a deterministically zero night and 20 pooled sites to route on. Which
mechanism do you expect to survive the confirmation, and does `sparse_moe` change your
prediction now that it can search narrower experts?

**E2 -- widen the space.** Add `w` to the search space (`"w": Choice([144, 288, 576])` in
`SPACE`, and give the study another name) instead of holding it fixed. First count: how many
discrete combinations does the space have now, and how many trials would cover a tenth of
them? Then run it with the same 36 trials. Does the best window interact with the best
mechanism, or are they independent? The parallel-categories plot in
`1_representation_and_head/04_search` is the tool for exactly this question.

**E3 -- tune the control.** The `deeper ~ moe` row of 4.4 runs at `deeper`'s own best `lr`,
not at one found for its larger width. Give it a search of its own (`mechanism` fixed to
`deeper`, `hidden` drawn around 161, `lr` drawn) with as many trials as `moe` had, then confirm
its best trial as in 4.4. Predict first: does the gap of 0.0027 (standard error 0.0013)
close?

**E4 -- your own dataset, your own specialisation question.** This search space is written
to be reusable as it stands: swap `load_solar_windows` for a loader over a different series of
the Monash forecasting archive, the public collection `solar_10_minutes` comes from (or your
own data), keep everything else, and re-run. Before you look at the result,
decide *why* you expect routing to help or not: does the series contain distinct regimes that
a handful of experts could divide up, or is it one homogeneous process end to end, like
electricity demand? The search tells you whether the bet paid off; only you can say in advance
whether the data gave it anything to win on.

**E5 -- read another schedule.** Give `lr_run` a `StepLR` instead of the cosine
(`scheduler=torch.optim.lr_scheduler.StepLR`, `scheduler_kwargs={"step_size": 5, "gamma": 0.3}`).
Before you run it, draw its `lr` per epoch from a start of 3e-3, and the update size you
expect beside it; then compare with what the optimizer reports. Second part: does the
chainsaw need a longer budget? Run the cosine arms for `EPOCHS` of 5, 15 and 45 with starts
1e-3 and 3e-3, and predict first at which budget, if any, the bigger start ends below the
smaller one.

**E6 -- grace period, arrival order and budget.** Run 4.5 with `GRACE` of 1, 3 and 5, each as
a study of its own name, and once with `n_startup_trials=20` so that more random trials arrive
before TPE takes over. Use the rung tables to see whom each stopped trial was compared with. Does a
pruned search train a `moe` to the end in any setting, and does the best score at a fixed
budget improve?

**E7 -- your own bound.** Name a use for this forecast and the bound it sets: a memory
budget, a speed in windows per second on your own machine, or the parameters `sparse_moe`
*uses* per window instead of the ones it trains. Measure what the front needs on your
machine, choose under your bound, and confirm the choice over new seeds as 4.4 does. How far
is its val MAE from the best model's, beside the standard error of that difference, and how
close does a faster or smaller model come?